In [1]:
from tensorflow import keras
import os
from PIL import Image

I0000 00:00:1780405113.894661    7281 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780405114.451820    7281 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780405116.174792    7281 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [31]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()

In [32]:
divisions = ['train','validation','others']
categories = ['airplane','ship','truck']

In [33]:
if not os.path.exists('./cifar10'):
	for division in divisions:
		for category in categories:
			os.makedirs(f'./cifar10/{division}/{category}')

In [34]:
def make_image_folder(label, category_name, ntrain, nvalidation):
	print(f'Making folder: label={label}, category_name={category_name},ntrain={ntrain},nvalidation={nvalidation}')
	count=0
	for i in range(train_images.shape[0]):
		if train_labels[i,0]==label:
			count+=1
			if count<=ntrain:
				division='train'
			elif count<=ntrain+nvalidation:
				division='validation'
			else:
				division='others'
			img = Image.fromarray(train_images[i])
			img.save(f'./cifar10/{division}/{category_name}/{i}.png')

In [35]:
make_image_folder(0,'airplane',800,200)
make_image_folder(8,'ship',800,200)
make_image_folder(9,'truck',800,200)

Making folder: label=0, category_name=airplane,ntrain=800,nvalidation=200
Making folder: label=8, category_name=ship,ntrain=800,nvalidation=200
Making folder: label=9, category_name=truck,ntrain=800,nvalidation=200


In [36]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import image_dataset_from_directory

In [37]:
categories = ['airplane','ship','truck']
ncategories = len(categories)

In [56]:
inputs=keras.Input(shape=(32,32,3))
x=layers.Rescaling(1./255)(inputs)
x=layers.Conv2D(filters=32,kernel_size=3,activation='relu')(x)
x=layers.Conv2D(filters=32,kernel_size=3,activation='relu')(x)
x=layers.MaxPooling2D(pool_size=2)(x)
x=layers.Dropout(0.25)(x)
x=layers.Conv2D(filters=32,kernel_size=3,activation='relu')(x)
x=layers.Conv2D(filters=32,kernel_size=3,activation='relu')(x)
x=layers.MaxPooling2D(pool_size=2)(x)
x=layers.Dropout(0.25)(x)
x=layers.Flatten()(x)
x=layers.Dense(512,activation='relu')(x)
x=layers.Dropout(0.5)(x)
outputs=layers.Dense(units=ncategories,activation='softmax')(x)
model=keras.Model(inputs=inputs,outputs=outputs)

In [57]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [58]:
train_dataset = image_dataset_from_directory('./cifar10/train', image_size=(32,32), batch_size=32)
validation_dataset = image_dataset_from_directory('./cifar10/validation', image_size=(32,32), batch_size=32)
callbacks = [keras.callbacks.ModelCheckpoint(filepath='./dsx06_2.keras', save_best_only=True, monitor="val_loss")]

Found 2400 files belonging to 3 classes.
Found 600 files belonging to 3 classes.


In [59]:
history = model.fit(
    train_dataset,epochs=1,validation_data=validation_dataset,callbacks=callbacks
)

75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.4446 - loss: 1.0345 - val_accuracy: 0.5517 - val_loss: 0.9355


In [60]:
from tensorflow import keras
import numpy as np
from PIL import Image

In [61]:
model=keras.models.load_model('./dsx06_2.keras')

In [62]:
mytestimages=['./cifar10/train/airplane/557.png','./cifar10/train/ship/5675.png','./cifar10/train/truck/6178.png',
              './cifar10/others/airplane/11208.png','./cifar10/others/ship/32852.png','./cifar10/others/truck/42042.png']
n=len(mytestimages)
test=np.zeros((n,32,32,3))
for i in range(n):
  img=Image.open(mytestimages[i])
  test[i]=np.array(img)

In [63]:
results = model.predict(test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step


In [64]:
np.set_printoptions(precision=6, floatmode='fixed', suppress=True)
print(results)

[[0.522540 0.433359 0.044100]
 [0.454509 0.510136 0.035355]
 [0.235763 0.192056 0.572181]
 [0.635678 0.346226 0.018097]
 [0.136999 0.259496 0.603505]
 [0.234868 0.219541 0.545591]]
